In [1]:
import os
import pandas as pd
from qiime2 import Visualization
from qiime2 import Artifact
import matplotlib.pyplot as plt 

%matplotlib inline

In [2]:
data_dir = '/home/jovyan/FunGut/data'

Define input file locations: 

In [3]:
reference_fasta = f"{data_dir}/raw/ITS_database.fasta"
clustered_seqs = f"{data_dir}/processed_ignored/fungut_seqs_closed_ref_ITS.qza"
clustered_table = f"{data_dir}/processed_ignored/fungut_table_closed_ref_ITS.qza"

Define the output file location:

In [4]:
taxonomy_output = f"{data_dir}/processed_ignored/fungut_taxonomy_closed_ref_ITS.qza"
taxonomy_visualization = f"{data_dir}/processed/fungut_taxonomy_closed_ref_ITS.qzv"

Inspect the FASTA headers to check for taxonomy labels and how they are separated from reference IDs:

In [5]:
! head -n 4 "$reference_fasta"

head: cannot open '/home/jovyan/FunGut/data/raw/ITS_database.fasta' for reading: No such file or directory


Read the reference FASTA into a pandas DataFrame using | as the column separator. Header lines of the FASTA file are divided into reference accession, taxonomy, and UNITE identifier.

In [6]:
reference = pd.read_csv(reference_fasta, sep="|", header=None)

FileNotFoundError: [Errno 2] No such file or directory: '/home/jovyan/FunGut/data/raw/ITS_database.fasta'

In [ ]:
reference.head()

Keep only the header rows:

In [ ]:
condition = reference[0].str.startswith(">")
headers = reference[condition].copy()
headers.head()

Extract the taxonomic labels from column 1 into a pandas Series:

In [ ]:
taxonomy = headers[1].copy()

Create feature IDs by joining the reference accession and kingdom label with |, without the leading >, so they match the IDs in our clustered table:

In [ ]:
feature_ids = headers[0].str.removeprefix(">") + "|" + headers[1].str.split(";").str[0]

Create a table linking each feature ID to its taxonomic label, keeping the labels in a separate column so QIIME 2 can use them:

In [ ]:
taxonomy_table = pd.DataFrame({"Feature ID": feature_ids, "Taxon": taxonomy})
taxonomy_table.head()

Convert the count table from closed-reference clustering into a pandas DataFrame so we can match its feature IDs to the taxonomic labels in our reference database:

In [ ]:
table_artifact = Artifact.load(clustered_table)
count_table = table_artifact.view(pd.DataFrame)
count_table.iloc[:3, :3] # visualize first three rows & columns 

Collect the feature IDs from the clustered count table so we can afterwards filter the reference taxonomy entries relevant to our samples:

In [ ]:
clustered_ids = count_table.columns

Select rows from the reference taxonomy table whose feature IDs match those in our clustered feature table.

-> Reference IDs: taxonomy_table["Feature ID"]  
-> Clustered feature IDs: clustered_ids

In [ ]:
condition = taxonomy_table["Feature ID"].isin(clustered_ids)
filtered_taxonomy_table = taxonomy_table[condition].copy()
filtered_taxonomy_table.iloc[:5, ]

Check that every clustered feature ID has a matching entry in the filtered taxonomy table (from the reference database):

In [ ]:
missing_ids = clustered_ids.difference(filtered_taxonomy_table["Feature ID"])
print(f"Features without taxonomy: {len(missing_ids)}")

Save the filtered taxonomy table as a TSV file for import into QIIME 2:

In [ ]:
taxonomy_tsv = f"{data_dir}/processed_ignored/fungut_taxonomy_closed_ref_ITS.tsv"

filtered_taxonomy_table.to_csv(
    taxonomy_tsv, sep="\t", index=False, header=False
)

Import the saved taxonomy table as a QIIME 2 artifact:

In [ ]:
! qiime tools import \
    --type 'FeatureData[Taxonomy]' \
    --input-format HeaderlessTSVTaxonomyFormat \
    --input-path "$taxonomy_tsv" \
    --output-path "$taxonomy_output"

Visualize the taxonomy table to analyze which fungal names are linked to our clustered features:

In [ ]:
! qiime metadata tabulate \
    --m-input-file "$taxonomy_output" \
    --o-visualization "$taxonomy_visualization"